# 00 — Setup environnement (Phase 0)

Notebook à ouvrir directement dans **Google Colab** (GPU T4 gratuit). Correspond aux étapes 3 à 14 de la Phase 0 décrite dans `docs/WORKFLOW.md`.

**Avant de commencer :** menu *Exécution* → *Modifier le type d'exécution* → Accélérateur matériel : **T4 GPU**.

## Étape 4 — Vérifier le GPU alloué

In [ ]:
!nvidia-smi

## Étape 5 — Monter Google Drive
Sert à sauvegarder les checkpoints/logs, qui doivent survivre à la déconnexion de la session.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Étape 6 — Créer l'arborescence sur Drive
Uniquement `checkpoints/`, `logs/`, `submissions/` : les données (`data/`) viennent directement du dépôt Git cloné ci-dessous, pas besoin de les dupliquer sur Drive.

In [ ]:
import os

PROJECT_DIR = '/content/drive/MyDrive/gemmafro-e2b'
for sub in ['checkpoints', 'logs', 'submissions']:
    os.makedirs(f'{PROJECT_DIR}/{sub}', exist_ok=True)

print(os.listdir(PROJECT_DIR))

## Étape 7 — Cloner le dépôt du projet (contient `data/`, `docs/`, `notebooks/`)

In [ ]:
import os

REPO_DIR = '/content/gemmafro-e2b'
if not os.path.isdir(REPO_DIR):
    !git clone https://github.com/andilMc/gemmafro-e2b.git {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

DATA_DIR = f'{REPO_DIR}/data'
print(os.listdir(DATA_DIR))

## Étape 8 — Installer les dépendances
À refaire à chaque nouvelle session (l'environnement Colab n'est pas persistant).

In [ ]:
!pip install -q -U transformers accelerate peft bitsandbytes trl datasets rouge-score pandas

## Étape 9-10 — Token Hugging Face (stocké dans les Colab Secrets)
Icône clé 🔑 dans la barre latérale → *Add new secret* → nom `HF_TOKEN`, valeur = token créé sur `huggingface.co/settings/tokens` → activer *Notebook access*.

In [ ]:
from google.colab import userdata
from huggingface_hub import login

login(token=userdata.get('HF_TOKEN'))

## Étape 11 — Licence du modèle
⚠️ Avant d'exécuter la cellule suivante : ouvrir la page du modèle exact sur Hugging Face et cliquer *Agree and access repository*. Vérifier aussi le nom exact du checkpoint (`MODEL_NAME` ci-dessous est un exemple à confirmer — cf. avertissement en tête de `docs/WORKFLOW.md`).

## Étape 12 — Test de chargement du modèle en 4-bit (QLoRA)

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_NAME = "google/gemma-3n-E2B-it"  # ⚠️ à remplacer par le nom exact du checkpoint vérifié sur HF

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,   # fp16, pas bf16 : T4 ne le supporte pas efficacement
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
)
print(f"Empreinte mémoire : {model.get_memory_footprint() / 1e9:.2f} Go")

## Étape 13 — Test rapide de génération (valider le chat template)

In [ ]:
messages = [{"role": "user", "content": "Bonjour, peux-tu répondre en français ?"}]
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
output = model.generate(**inputs, max_new_tokens=50)
print(tokenizer.decode(output[0], skip_special_tokens=True))

## Étape 14 — Vérifier l'accès aux données du dépôt cloné

In [ ]:
import pandas as pd

train = pd.read_csv(f'{DATA_DIR}/Train.csv')
val   = pd.read_csv(f'{DATA_DIR}/Val.csv')
test  = pd.read_csv(f'{DATA_DIR}/Test.csv')

print('Train:', train.shape, '| Val:', val.shape, '| Test:', test.shape)
train.head(3)

---
**Si toutes les cellules ci-dessus s'exécutent sans erreur : l'environnement est prêt.** Étape suivante : Phase 1 — Analyse exploratoire des données (voir `docs/WORKFLOW.md`).